# الدرس 10 - وكلاء الذكاء الاصطناعي في الإنتاج

في هذا الدرس ستتعلم **أنماط الإنتاج** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل Microsoft Agent (بايثون). نغطي:

- **الرصد** — إضافة توقيت وتسجيل التفاعلات مع الوكيل
- **التقييم** — استخدام وكيل مقيم لتقييم جودة الاستجابة
- **إدارة التكلفة** — استراتيجيات لتحسين عدد التوكنات واختيار النماذج

السيناريو هو **وكيل سفر** يساعد المستخدمين في تخطيط الرحلات، مع طبقة رصد وتقييم مضافة فوق ذلك.


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## اعتبارات الإنتاج

نقل وكلاء الذكاء الاصطناعي من النماذج الأولية إلى الإنتاج يتطلب اهتمامًا دقيقًا بثلاث ركائز:

1. **الرصد** — تحتاج إلى رؤية ما يفعله الوكيل، ومدة استغراقه، والأدوات التي يستخدمها. بدون التتبع والتسجيل، يكون من المستحيل تقريبًا تصحيح مشكلات الإنتاج.

2. **التقييم** — تضمن فحوصات الجودة الآلية بقاء استجابات الوكيل دقيقة وشاملة ومفيدة مع مرور الوقت. يمكن لوكيل التقييم أن يقيم الاستجابات مقابل المعايير المحددة.

3. **إدارة التكاليف** — يؤثر استخدام الرموز مباشرة على التكلفة. تساعد استراتيجيات مثل تحسين المطالبات، اختيار النماذج، والتخزين المؤقت في الحفاظ على النفقات تحت السيطرة دون التضحية بالجودة.


## بناء وكيل يمكن ملاحظته

نعرف أدوات السفر ونغلف استدعاء الوكيل بالتوقيت حتى نتمكن من مراقبة الكمون. في الإنتاج، ستدمج مع OpenTelemetry أو نظام تتبع مشابه.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## أنماط التقييم

النمط الشائع في الإنتاج هو استخدام وكيل ثانٍ كـ **مُقَيِّم**. يقوم المُقَيِّم بتقييم استجابة الوكيل الأساسي مقارنة بمعايير محددة مسبقًا مثل الشمول والدقة والفائدة.

هذا يُمكِّن من:
- بوابات جودة آلية قبل وصول الاستجابات إلى المستخدمين
- الكشف عن التراجعات عند تغيير الموجهات أو النماذج
- المراقبة المستمرة لأداء الوكيل على مدى الوقت


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتيجيات إدارة التكاليف

التحكم في التكاليف أمر بالغ الأهمية لوكلاء الذكاء الاصطناعي الإنتاجيين. فيما يلي استراتيجيات رئيسية:

| الاستراتيجية | الوصف |
|---|---|
| **تحسين المُحفزات** | اجعل تعليمات النظام موجزة. أزل السياق المكرر لتقليل الرموز المُدخلة. |
| **اختيار النموذج** | استخدم نماذج أصغر وأرخص (مثل GPT-4o-mini) للمهام البسيطة مثل التصنيف أو الاستخراج، وخصص النماذج الأكبر للمنطق المعقد. |
| **التخزين المؤقت** | خزّن نتائج الأدوات والاستعلامات المتكررة لتجنب نداءات API المكررة. |
| **ميزانيات الرموز** | ضع حدود `max_tokens` لمنع الردود الطويلة غير المتوقعة. |
| **التجميع** | جمع استعلامات المستخدم المتعددة في نداء API واحد حيثما أمكن. |

في الممارسة العملية، يعمل النهج المتدرج بشكل جيد: قم بإرسال الطلبات البسيطة إلى نموذج سريع وغير مكلف وتصعيد الاستعلامات المعقدة فقط إلى نموذج أكثر قدرة.


## الملخص

في هذا الدرس تعلمت كيفية:

1. **إضافة القابلية للملاحظة** لتفاعلات الوكيل مع التوقيت وتسجيل الدخول، مما يؤسس لرصد وتتبع الأداء.
2. **تقييم استجابات الوكيل** تلقائيًا باستخدام وكيل تقييم يقوم بتسجيل الكمال والدقة والفائدة.
3. **إدارة التكاليف** من خلال تحسين المطالبات، اختيار النماذج، التخزين المؤقت، وميزانيات الرموز.

تساعدك هذه الأنماط الإنتاجية على ضمان أن وكلاء الذكاء الاصطناعي الخاصين بك موثوقين، قابليين للقياس، وفعّالين من حيث التكلفة على نطاق واسع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
